# Clustering Tutorial

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/akekapong78/ai-competition/blob/main/05-clustering/clustering_tutorial.ipynb)

**เป้าหมาย:** เรียนรู้ clustering algorithms หลัก + ใช้งานจริงกับ image และ time series

| Algorithm | ใช้เมื่อ |
|-----------|----------|
| K-Means | รู้จำนวน cluster, ข้อมูล spherical |
| DBSCAN | ไม่รู้จำนวน cluster, มี noise/outlier |
| Hierarchical | อยากดู dendrogram, dataset เล็ก |

> **Colab:** Runtime → Change runtime type → T4 GPU (ถ้าใช้ image section)

## 0. Setup

In [1]:
!pip install scikit-learn matplotlib seaborn pandas numpy scipy -q

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score
from sklearn.datasets import make_blobs, make_moons
from scipy.cluster.hierarchy import dendrogram, linkage

plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.size'] = 12
print('Setup done')

Setup done


## 1. Clustering คืออะไร?

**Supervised Learning:** มี label → model เรียนรู้จาก label  
**Clustering (Unsupervised):** ไม่มี label → model หา pattern เอง

```
ข้อมูล: [จุดA, จุดB, จุดC, จุดD, จุดE]
→ Clustering หาว่า จุดไหนคล้ายกัน → รวมเป็น group
→ Cluster 1: [A, C, E]  Cluster 2: [B, D]
```

**ใช้ใน competition:**
- Image: แบ่ง pixel ตามสี (color segmentation)
- Forecasting: จัดกลุ่ม load pattern ที่คล้ายกัน
- อธิบาย pattern ใน data ก่อน build model

## 2. K-Means Clustering

**Algorithm:**
1. เลือก K จุดสุ่มเป็น centroid
2. แต่ละจุดไปหา centroid ที่ใกล้ที่สุด
3. คำนวณ centroid ใหม่ (mean ของแต่ละ cluster)
4. ทำซ้ำจนไม่เปลี่ยน

**ข้อจำกัด:** ต้องกำหนด K ก่อน → ใช้ Elbow Method ช่วย

In [ ]:
# สร้าง synthetic data — 4 cluster ชัดเจน
X, y_true = make_blobs(n_samples=400, centers=4, cluster_std=0.8, random_state=42)

# Scale ก่อนเสมอ — K-Means ใช้ distance → scale สำคัญมาก
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Train K-Means (K=4)
km = KMeans(n_clusters=4, random_state=42, n_init=10)
labels = km.fit_predict(X_scaled)

# Plot
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.scatter(X[:, 0], X[:, 1], c='gray', alpha=0.5, s=20)
ax1.set_title('Data (no labels)')

colors = ['#e74c3c', '#3498db', '#2ecc71', '#f39c12']
for k in range(4):
    mask = labels == k
    ax2.scatter(X[mask, 0], X[mask, 1], c=colors[k], label=f'Cluster {k}', s=20)

# Plot centroids
centroids = scaler.inverse_transform(km.cluster_centers_)
ax2.scatter(centroids[:, 0], centroids[:, 1], c='black', marker='X', s=200, zorder=5, label='Centroids')
ax2.set_title('K-Means Result (K=4)')
ax2.legend()

plt.tight_layout()
plt.show()

score = silhouette_score(X_scaled, labels)
print(f'Silhouette Score: {score:.3f}  (ยิ่งใกล้ 1 ยิ่งดี — cluster ชัดเจน)')

## 3. Elbow Method — หาจำนวน K ที่เหมาะ

**Inertia** = sum of squared distances ของทุกจุดถึง centroid  
→ ยิ่ง K มาก inertia ยิ่งลด แต่ถึงจุดหนึ่งลดน้อยลง = **Elbow point**

In [ ]:
inertias   = []
silhouettes = []
K_range    = range(2, 11)

for k in K_range:
    km_k = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels_k = km_k.fit_predict(X_scaled)
    inertias.append(km_k.inertia_)
    silhouettes.append(silhouette_score(X_scaled, labels_k))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))

ax1.plot(list(K_range), inertias, 'o-', color='steelblue')
ax1.axvline(x=4, color='red', linestyle='--', label='Elbow (K=4)')
ax1.set_xlabel('K (number of clusters)')
ax1.set_ylabel('Inertia')
ax1.set_title('Elbow Method')
ax1.legend()
ax1.grid(alpha=0.3)

ax2.plot(list(K_range), silhouettes, 'o-', color='green')
ax2.axvline(x=silhouettes.index(max(silhouettes)) + 2, color='red', linestyle='--',
            label=f'Best K={silhouettes.index(max(silhouettes)) + 2}')
ax2.set_xlabel('K')
ax2.set_ylabel('Silhouette Score')
ax2.set_title('Silhouette Score (ยิ่งสูงยิ่งดี)')
ax2.legend()
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()
print('เลือก K ที่ elbow (inertia ลดช้าลง) และ silhouette สูงสุด')

## 4. DBSCAN — Density-Based Clustering

**ข้อดีกว่า K-Means:**
- ไม่ต้องกำหนด K
- หา cluster รูปทรงแปลก (non-spherical) ได้
- ระบุ noise/outlier ได้ (label = -1)

**Parameters:**
- `eps`: รัศมีที่ถือว่า "ใกล้" (ลองหลายค่า)
- `min_samples`: จำนวนจุดขั้นต่ำในรัศมีที่เป็น core point

In [ ]:
# DBSCAN เก่งกับ data รูปทรงแปลก เช่น make_moons
X_moon, _ = make_moons(n_samples=300, noise=0.08, random_state=42)
X_moon_scaled = StandardScaler().fit_transform(X_moon)

# K-Means ทำได้ไม่ดีกับ moons
km_moon = KMeans(n_clusters=2, random_state=42, n_init=10)
labels_km = km_moon.fit_predict(X_moon_scaled)

# DBSCAN ทำได้ดีกว่า
db = DBSCAN(eps=0.2, min_samples=5)
labels_db = db.fit_predict(X_moon_scaled)
n_clusters_db = len(set(labels_db)) - (1 if -1 in labels_db else 0)
n_noise      = (labels_db == -1).sum()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.scatter(X_moon[:, 0], X_moon[:, 1], c=labels_km, cmap='RdBu', s=20)
ax1.set_title('K-Means (K=2) — ทำได้ไม่ดี')

# DBSCAN: -1 = noise → สีดำ
cmap = plt.cm.tab10
for lbl in set(labels_db):
    mask = labels_db == lbl
    color = 'black' if lbl == -1 else cmap(lbl / max(1, n_clusters_db))
    label = 'Noise' if lbl == -1 else f'Cluster {lbl}'
    ax2.scatter(X_moon[mask, 0], X_moon[mask, 1], c=[color]*mask.sum(), label=label, s=20)

ax2.set_title(f'DBSCAN — {n_clusters_db} clusters, {n_noise} noise points')
ax2.legend()

plt.tight_layout()
plt.show()

## 5. Hierarchical Clustering + Dendrogram

**Algorithm:** เริ่มจากแต่ละจุดเป็น cluster ของตัวเอง → รวมที่ใกล้ที่สุดทีละคู่  
→ ได้ **dendrogram** — ดูว่าจะตัดที่ระดับไหน = กี่ cluster

In [ ]:
# ใช้ subset เล็กๆ (hierarchical ช้ากับ data ใหญ่)
X_small = X[:80]
X_small_scaled = StandardScaler().fit_transform(X_small)

# Dendrogram
linked = linkage(X_small_scaled, method='ward')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

dendrogram(linked, ax=ax1, truncate_mode='lastp', p=20, leaf_rotation=45)
ax1.axhline(y=5, color='red', linestyle='--', label='cut at 5 → 4 clusters')
ax1.set_title('Dendrogram (Ward linkage)')
ax1.set_xlabel('Sample index')
ax1.set_ylabel('Distance')
ax1.legend()

# Hierarchical cluster result
hc = AgglomerativeClustering(n_clusters=4, linkage='ward')
labels_hc = hc.fit_predict(X_small_scaled)

for k in range(4):
    mask = labels_hc == k
    ax2.scatter(X_small[mask, 0], X_small[mask, 1], label=f'Cluster {k}', s=40)
ax2.set_title('Hierarchical Clustering (4 clusters)')
ax2.legend()

plt.tight_layout()
plt.show()

## 6. Application: Image Color Segmentation

**ใช้ K-Means แบ่ง pixel ตามสี** — ลด complexity ของภาพ  
ประโยชน์: preprocessing ก่อน object detection, ลด noise

In [ ]:
from sklearn.datasets import load_sample_image

# โหลด sample image (มาพร้อม scikit-learn)
china = load_sample_image('china.jpg')
print(f'Image shape: {china.shape}')  # (H, W, 3) — RGB

# Resize เพื่อความเร็ว
from PIL import Image as PILImage
img_small = np.array(PILImage.fromarray(china).resize((200, 150)))

# แปลง pixel เป็น (N, 3) — แต่ละ row คือ [R, G, B]
h, w, c = img_small.shape
pixels = img_small.reshape(-1, 3).astype(np.float32) / 255.0
print(f'Pixels: {pixels.shape}')  # (30000, 3)

# K-Means ด้วย K=8 สี
n_colors = 8
km_img = KMeans(n_clusters=n_colors, random_state=42, n_init=3)
km_img.fit(pixels)

# แทนทุก pixel ด้วยสี centroid ของ cluster นั้น
compressed = km_img.cluster_centers_[km_img.labels_]
img_compressed = (compressed.reshape(h, w, 3) * 255).astype(np.uint8)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ax1.imshow(img_small)
ax1.set_title('Original')
ax1.axis('off')

ax2.imshow(img_compressed)
ax2.set_title(f'Compressed ({n_colors} colors via K-Means)')
ax2.axis('off')

plt.tight_layout()
plt.show()

orig_mb = h * w * 3 / 1024**2
print(f'Original: {orig_mb:.2f} MB → compressed to {n_colors} unique colors')

## 7. Application: Time Series Pattern Clustering

**จัดกลุ่ม load pattern รายวัน** — ดูว่าวันไหนมีพฤติกรรมคล้ายกัน  
ใช้ใน forecasting: สร้าง feature `cluster_type` ให้ model รู้ว่าวันนี้เป็น pattern แบบไหน

In [ ]:
np.random.seed(42)

# สร้าง synthetic daily solar profiles (24 ชั่วโมง)
n_days  = 200
hours   = np.arange(24)
profiles = []

for i in range(n_days):
    day_type = np.random.choice(['sunny', 'cloudy', 'rainy', 'weekend_sunny'], 
                                 p=[0.4, 0.3, 0.2, 0.1])
    if day_type == 'sunny':
        p = 10 * np.maximum(0, np.sin(np.pi * (hours - 6) / 12))
        p += np.random.normal(0, 0.3, 24)
    elif day_type == 'cloudy':
        p = 5 * np.maximum(0, np.sin(np.pi * (hours - 6) / 12))
        p += np.random.normal(0, 0.5, 24)
    elif day_type == 'rainy':
        p = 2 * np.maximum(0, np.sin(np.pi * (hours - 6) / 12))
        p += np.random.normal(0, 0.2, 24)
    else:  # weekend sunny — คนใช้ไฟน้อยกว่า
        p = 8 * np.maximum(0, np.sin(np.pi * (hours - 7) / 11))
        p += np.random.normal(0, 0.4, 24)
    profiles.append(np.maximum(0, p))

profiles = np.array(profiles)
profiles_scaled = StandardScaler().fit_transform(profiles)

# K-Means หา pattern
km_ts = KMeans(n_clusters=4, random_state=42, n_init=10)
cluster_labels = km_ts.fit_predict(profiles_scaled)

# Plot mean profile ของแต่ละ cluster
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
cluster_names = ['Cluster 0', 'Cluster 1', 'Cluster 2', 'Cluster 3']

for k, ax in enumerate(axes.flat):
    mask = cluster_labels == k
    cluster_profiles = profiles[mask]
    
    # Plot individual profiles (faint)
    for prof in cluster_profiles[:10]:
        ax.plot(hours, prof, alpha=0.2, color='steelblue', linewidth=0.8)
    
    # Plot mean profile
    ax.plot(hours, cluster_profiles.mean(axis=0), color='red', linewidth=2.5, label='Mean')
    ax.set_title(f'{cluster_names[k]} — {mask.sum()} วัน')
    ax.set_xlabel('Hour')
    ax.set_ylabel('Solar (MW)')
    ax.legend()
    ax.grid(alpha=0.3)

plt.suptitle('Daily Solar Profile Clusters', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

sil = silhouette_score(profiles_scaled, cluster_labels)
print(f'Silhouette Score: {sil:.3f}')

## 8. Algorithm Comparison

เปรียบเทียบทั้ง 3 algorithm บน dataset เดียวกัน

In [ ]:
datasets = {
    'Blobs (4 cluster)': make_blobs(n_samples=300, centers=4, cluster_std=0.8, random_state=42)[0],
    'Moons (non-spherical)': make_moons(n_samples=300, noise=0.08, random_state=42)[0],
}

algorithms = {
    'K-Means (K=4)':    KMeans(n_clusters=4, random_state=42, n_init=10),
    'DBSCAN':           DBSCAN(eps=0.3, min_samples=5),
    'Hierarchical (4)': AgglomerativeClustering(n_clusters=4, linkage='ward'),
}

fig, axes = plt.subplots(len(datasets), len(algorithms), figsize=(16, 8))

for row, (ds_name, X_ds) in enumerate(datasets.items()):
    X_ds_scaled = StandardScaler().fit_transform(X_ds)
    for col, (alg_name, alg) in enumerate(algorithms.items()):
        ax = axes[row, col]
        labels_c = alg.fit_predict(X_ds_scaled)
        
        n_cls = len(set(labels_c)) - (1 if -1 in labels_c else 0)
        noise = (labels_c == -1).sum()
        
        ax.scatter(X_ds[:, 0], X_ds[:, 1], c=labels_c, cmap='tab10', s=15, alpha=0.8)
        title = f'{alg_name}\n{n_cls} clusters'
        if noise > 0:
            title += f', {noise} noise'
        ax.set_title(title, fontsize=10)
        ax.set_xlabel(ds_name if col == 0 else '')
        ax.tick_params(left=False, bottom=False, labelleft=False, labelbottom=False)

plt.suptitle('Algorithm Comparison', fontsize=13)
plt.tight_layout()
plt.show()

## Summary

| Algorithm | จำนวน cluster | รูปทรง | Outlier | Speed |
|-----------|--------------|--------|---------|-------|
| **K-Means** | ต้องกำหนด | Spherical เท่านั้น | ไม่ handle | เร็วมาก |
| **DBSCAN** | อัตโนมัติ | ทุกรูปทรง | ระบุได้ (label=-1) | ปานกลาง |
| **Hierarchical** | เลือกจาก dendrogram | ปานกลาง | ไม่ handle | ช้า (O(n²)) |

### เลือกยังไง?

```
รู้ K แน่นอน + data spherical     → K-Means
ไม่รู้ K + data รูปแปลก + มี noise → DBSCAN
อยากดู hierarchy ของ data          → Hierarchical
```

### Evaluation

```python
silhouette_score(X, labels)   # -1 ถึง 1, ยิ่งสูงยิ่งดี
km.inertia_                   # K-Means: ยิ่งต่ำยิ่งดี (ใช้กับ Elbow)
```